# Sub-basin zonal stats → `outputs/subbasin_zonal_<dataset>.csv`

**Kernel:** `rasterio`, `numpy` (no Fiona/rasterstats).

**Requires GeoTIFFs** under `rasters/<dataset>/geotiff/<prefix>_YYYY.tif`.

Set `DATASET_FILTER` in the next cell to `"hilda"` (or another key) to process one dataset only, or `None` for all.

In [1]:
from pathlib import Path


def find_data_root(start: Path) -> Path:
    for d in [start, *start.parents]:
        if (d / "lt_subbasins.json").is_file():
            return d
    raise FileNotFoundError("lt_subbasins.json not found — set DATA_ROOT manually.")


DATA_ROOT = find_data_root(Path.cwd())
SUBBASINS_PATH = DATA_ROOT / "lt_subbasins.json"
OUT_DIR = DATA_ROOT / "outputs"

DATASETS = {
    "hilda": (DATA_ROOT / "rasters" / "hilda" / "geotiff", "hilda"),
    "lucas": (DATA_ROOT / "rasters" / "lucas" / "geotiff", "lucas"),
    "hyde": (DATA_ROOT / "rasters" / "hyde" / "geotiff", "hyde"),
    "luh2": (DATA_ROOT / "rasters" / "luh2" / "geotiff", "luh2"),
    "corine": (DATA_ROOT / "rasters" / "corine" / "geotiff", "corine"),
}

# None = all datasets; or e.g. "hilda"
DATASET_FILTER = None

print("DATA_ROOT:", DATA_ROOT)
print("Filter:", DATASET_FILTER or "all")

DATA_ROOT: C:\Users\matas\Desktop\LEI\Data
Filter: all


In [2]:
import json

import numpy as np
import rasterio
from rasterio.mask import mask


def load_features():
    with open(SUBBASINS_PATH, "r", encoding="utf-8") as f:
        fc = json.load(f)
    return fc.get("features") or []


def list_years(geotiff_dir: Path, prefix: str):
    if not geotiff_dir.is_dir():
        return []
    years = []
    for p in geotiff_dir.glob(f"{prefix}_*.tif"):
        try:
            years.append(int(p.stem.split("_")[-1]))
        except (ValueError, IndexError):
            continue
    return sorted(set(years))


def count_classes_in_basin(src, feature: dict):
    geom = feature.get("geometry")
    if not geom:
        return {}
    try:
        out_image, _ = mask(src, [geom], crop=True, nodata=0, indexes=1)
    except ValueError:
        return {}
    data = out_image[0] if out_image.ndim == 3 else out_image
    valid = data[(data >= 1) & (data <= 5)]
    if valid.size == 0:
        return {}
    uniq, cnts = np.unique(valid, return_counts=True)
    return {int(u): int(c) for u, c in zip(uniq, cnts, strict=False)}


def process_dataset(ds_key: str, features: list):
    folder, prefix = DATASETS[ds_key]
    years = list_years(folder, prefix)
    if not years:
        print(f"[{ds_key}] No GeoTIFF under {folder} — skip.")
        return
    out_path = OUT_DIR / f"subbasin_zonal_{ds_key}.csv"
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    rows_written = 0
    with open(out_path, "w", encoding="utf-8") as out:
        out.write("year,basin_index,class_id,count\n")
        for year in years:
            tif = folder / f"{prefix}_{year}.tif"
            if not tif.is_file():
                continue
            print(f"  [{ds_key}] {year} …")
            with rasterio.open(tif) as src:
                for basin_index, feat in enumerate(features):
                    counts = count_classes_in_basin(src, feat)
                    for cid in range(1, 6):
                        c = counts.get(cid, 0)
                        if c > 0:
                            out.write(f"{year},{basin_index},{cid},{c}\n")
                            rows_written += 1
    print(f"[{ds_key}] Wrote {out_path} ({rows_written} rows)")

print("Ready.")

Ready.


In [3]:
features = load_features()
if not features:
    raise RuntimeError("No features in sub-basins GeoJSON")
print(f"Loaded {len(features)} basins")

keys = [DATASET_FILTER] if DATASET_FILTER else list(DATASETS.keys())
for ds_key in keys:
    if ds_key not in DATASETS:
        print("Unknown dataset:", ds_key)
        continue
    print(f"\n=== {ds_key} ===")
    process_dataset(ds_key, features)

print("\nDone.")

Loaded 19 basins

=== hilda ===
  [hilda] 1910 …
  [hilda] 1911 …
  [hilda] 1912 …
  [hilda] 1913 …
  [hilda] 1914 …
  [hilda] 1915 …
  [hilda] 1916 …
  [hilda] 1917 …
  [hilda] 1918 …
  [hilda] 1919 …
  [hilda] 1920 …
  [hilda] 1921 …
  [hilda] 1922 …
  [hilda] 1923 …
  [hilda] 1924 …
  [hilda] 1925 …
  [hilda] 1926 …
  [hilda] 1927 …
  [hilda] 1928 …
  [hilda] 1929 …
  [hilda] 1930 …
  [hilda] 1931 …
  [hilda] 1932 …
  [hilda] 1933 …
  [hilda] 1934 …
  [hilda] 1935 …
  [hilda] 1936 …
  [hilda] 1937 …
  [hilda] 1938 …
  [hilda] 1939 …
  [hilda] 1940 …
  [hilda] 1941 …
  [hilda] 1942 …
  [hilda] 1943 …
  [hilda] 1944 …
  [hilda] 1945 …
  [hilda] 1946 …
  [hilda] 1947 …
  [hilda] 1948 …
  [hilda] 1949 …
  [hilda] 1950 …
  [hilda] 1951 …
  [hilda] 1952 …
  [hilda] 1953 …
  [hilda] 1954 …
  [hilda] 1955 …
  [hilda] 1956 …
  [hilda] 1957 …
  [hilda] 1958 …
  [hilda] 1959 …
  [hilda] 1960 …
  [hilda] 1961 …
  [hilda] 1962 …
  [hilda] 1963 …
  [hilda] 1964 …
  [hilda] 1965 …
  [hilda] 1966 …